# Amazon ML Challenge 2026 — Data and Pipeline Review

This notebook is a lightweight, streaming audit of the supplied TSV schemas and row distributions. It does not load the full records into pandas and does not query external sources. Run it from the repository root or the `notebooks/` directory.

The full files contain millions of rows each; the audit can take several minutes.


In [ ]:
from collections import Counter
import csv
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "dataset").exists():
    ROOT = ROOT.parent
DATA = ROOT / "dataset" / "student_resource" / "dataset"
FILES = [
    "train/train_source1.tsv",
    "train/train_source2.tsv",
    "train/train_source3.tsv",
    "train/train_ground_truth.tsv",
    "test/test_source1.tsv",
    "test/test_source2.tsv",
    "test/test_source3.tsv",
]
print("Repository:", ROOT)
print("Data directory exists:", DATA.is_dir())


In [ ]:
def profile_tsv(relative_path):
    path = DATA / relative_path
    row_count = 0
    empty_name = 0
    empty_address = 0
    countries = Counter()
    match_lengths = Counter()
    with path.open("r", encoding="utf-8-sig", newline="") as stream:
        reader = csv.DictReader(stream, delimiter="\t")
        headers = reader.fieldnames or []
        for row in reader:
            row_count += 1
            if "business_name" in row and not (row.get("business_name") or "").strip():
                empty_name += 1
            if "business_address" in row and not (row.get("business_address") or "").strip():
                empty_address += 1
            if "country" in row:
                countries[(row.get("country") or "").strip()] += 1
            if "matched_entity_ids" in row:
                value = (row.get("matched_entity_ids") or "").strip()
                match_lengths[len([item for item in value.split(",") if item.strip()]) if value else 0] += 1
    return {
        "file": relative_path,
        "rows": row_count,
        "columns": headers,
        "empty_names": empty_name,
        "empty_addresses": empty_address,
        "countries": countries,
        "ground_truth_match_lengths": match_lengths,
    }


In [ ]:
profiles = [profile_tsv(name) for name in FILES]
for result in profiles:
    print("\n", result["file"], "rows:", result["rows"])
    print("columns:", result["columns"])
    print("empty names / addresses:", result["empty_names"], "/", result["empty_addresses"])
    if result["countries"]:
        print("countries:", result["countries"])
    if result["ground_truth_match_lengths"]:
        print("ground-truth match lengths:", result["ground_truth_match_lengths"])


## Interpretation checklist

- Ensure readers use a tab separator and retain IDs as strings.
- Keep `country` open-set; test contains France.
- Evaluate candidate recall before tuning the pair classifier.
- Count every Source 1 entity in macro F₀.₅, including empty ground-truth lists.
- Do not treat notebook metrics as model results unless the actual pipeline and held-out split generated them.
